# TP1 - Import de formats et accès aux API
## Des petits scripts python pour nous aider

In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

### 0 - Lire un jeu de donnée au format CSV avec geopandas

#### TODO : Définir les paramètres associées aux chemins de fichier (utiliser Path)

In [ ]:
dossier_courant = 
print(f"Dossier de travail actuel : {dossier_courant}")

In [ ]:
dossier_donnees =
# On crée un dossier output/ dans le dossier data/ pour y mettre les fichiers générés par des scripts python
dossier_sortie = dossier_donnees / "output"
dossier_sortie.mkdir(parents=True, exist_ok=True)

In [ ]:
fichier_csv = 
colonne_wkt = "eca_h"
crs_source = "EPSG:2154"

In [ ]:
df = pd.read_csv(fichier_csv)
geometry = gpd.GeoSeries.from_wkt(df[colonne_wkt], crs=crs_source)

if colonne_wkt != "geometry":
    df = df.drop(columns=[colonne_wkt])

gdf = gpd.GeoDataFrame(df, geometry=geometry)

In [ ]:
gdf.head()

In [ ]:
print("Lignes, colonnes :", gdf.shape)
print("CRS :", gdf.crs)
print("Emprise :", gdf.total_bounds)
print("\nTypes de géométries :", gdf.geom_type.value_counts())
print("\nGéométries invalides :", (~gdf.is_valid).sum())

### 1 - Convertir vers un autre format de données

Exportez votre GeoDataFrame (gdf) dans les quatre formats suivants : GeoPackage, GeoParquet, GeoJSON et Shapefile.

- Pour GeoPackage, GeoJSON et Shapefile : utilisez la méthode .to_file() en précisant le chemin du fichier de sortie et le driver correspondant :

    ```py
    gdf.to_file(dossier_sortie / f"{nom_base}.geojson", driver="GeoJSON")
    ``` 

(Drivers à utiliser : "GPKG", "GeoJSON" et "ESRI Shapefile")

- Pour GeoParquet : utilisez la méthode spécifique .to_parquet() (qui ne nécessite pas de paramètre driver) 

In [ ]:
# TODO
nom_base = fichier_csv.stem






### 2 - Mesures et analyses des fichiers

In [ ]:
import time

In [ ]:
dossier_sortie_mesures = dossier_sortie / "mesures"
dossier_sortie_mesures.mkdir(parents=True, exist_ok=True)

In [ ]:
formats = {
    "Shapefile": (dossier_sortie_mesures / f"{nom_base}.shp", "ESRI Shapefile"),
    "GeoJSON": (dossier_sortie_mesures / f"{nom_base}.geojson", "GeoJSON"),
    "GeoPackage": (dossier_sortie_mesures / f"{nom_base}.gpkg", "GPKG"),
}

In [ ]:
rows = []
MB_WINDOWS = 1024 ** 2  # et non 1e6 pour des Mo
# l'explorateur windows appelle Mo ce qui est en réalité des Mio (Mébioctets) soit 1024*1024 octets
exts_shp = {".shp", ".shx", ".dbf", ".prj", ".cpg", ".sbn", ".sbx"}

for nom, (path, driver) in formats.items():
    t0 = time.perf_counter()
    gdf.to_file(path, driver=driver)
    t_write = time.perf_counter() - t0

    t0 = time.perf_counter()
    _ = gpd.read_file(path)
    t_read = time.perf_counter() - t0
    
    if nom == "Shapefile":
        size_mb = sum(f.stat().st_size for f in dossier_sortie_mesures.glob(f"{nom_base}.*") if f.suffix.lower() in exts_shp) / MB_WINDOWS
    else:
        size_mb = path.stat().st_size / MB_WINDOWS

    rows.append({
        "Format": nom,
        "Taille (Mo)": round(size_mb, 2),
        "Écriture (s)": round(t_write, 3),
        "Lecture (s)": round(t_read, 3),
    })

parquet_path = dossier_sortie_mesures / f"{nom_base}.parquet"
t0 = time.perf_counter()
gdf.to_parquet(parquet_path, index=False)
t_write = time.perf_counter() - t0

t0 = time.perf_counter()
_ = gpd.read_parquet(parquet_path)
t_read = time.perf_counter() - t0

rows.append({
    "Format": "GeoParquet",
    "Taille (Mo)": round(parquet_path.stat().st_size / MB_WINDOWS, 2),
    "Écriture (s)": round(t_write, 3),
    "Lecture (s)": round(t_read, 3),
})

In [ ]:
print(pd.DataFrame(rows).to_string(index=False))

### 3 - Récupérer des entitées dans OSM

In [ ]:
import osmnx as ox

In [ ]:
fichier_zone = dossier_donnees / "reserve.gpkg" # contour de la reserve
fichier_sortie = dossier_donnees / "poi.gpkg"
nom_couche_sortie = "all_poi"

zone = gpd.read_file(fichier_zone)
polygone_wgs84 = zone.to_crs("EPSG:4326").geometry.union_all()

#### TODO : Ajouter d'autres sortes de POI

In [ ]:
tags_poi = {
    "amenity": ["drinking_water", "", ""],
    "natural": ["spring", "", ""],
    "tourism": ["viewpoint", "", "", ""],
}

In [ ]:
poi = ox.features_from_polygon(polygone_wgs84, tags=tags_poi).to_crs("EPSG:2154")
poi_parc = gpd.clip(poi, zone)
cols = [c for c in ["name", "amenity", "natural", "tourism", "geometry"] if c in poi_parc.columns]
print(f"Nombre d'objets récupérés dans le parc : {len(poi_parc)}")
poi_parc.to_file(fichier_sortie, layer=nom_couche_sortie, driver="GPKG")